<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "120"
iso_country = "Cameroon"
year = 2023
import_portfolio = "h6_ag6_by_sea_M*.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_X*.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 48.800*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Cameroon': 

Number of trade records present in dataset: 37943
Number of partner countries present in dataset: 163
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$5665.9m
Total weight within the dataset: 7461.4 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,120,Cameroon,826,United Kingdom,Import,Sea,392210,"392210 | Plastics; baths, sho",3922,3922 | Sanitary ware; baths,39,39 | Plastics and article,3693,"Baths, wash-basins, lavatory pans and covers, ...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",1249.929812,NaN,1249.929812,kg,3282.853,3282.853,208.238307,7896.0,4906.345416,1021.689061,['Container'],['Container'],3.257277,3.244731,3.245663,3.245663,3.253658,3.253658,3.424396,6.482543,3.796016,3.476670,3.490739,3.600514,3.345099,6.183619,3.952979,3.625305,3.541626,3.723207,3.275613,6.844313,4.934517,4.314288,3.985759,3.892781,3.207090,7.244721,5.772691,5.409113,4.224981,4.191274,3.139829,7.365876,6.981270,7.007492,4.601890,4.411527,3.071279,7.270138,6.992989,7.463998,5.550840,4.494830,3.006547,6.413915,6.668964,7.076298,5.335733,4.549896,2.940986,5.748879,5.891919,6.965658,5.685667,4.585572,3693
1,1,120,Cameroon,804,Ukraine,Import,Sea,392210,"392210 | Plastics; baths, sho",3922,3922 | Sanitary ware; baths,39,39 | Plastics and article,3693,"Baths, wash-basins, lavatory pans and covers, ...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",5883.985627,NaN,5883.985627,kg,7705.218,7705.218,980.272005,10263.0,6377.130573,6251.322576,['Container'],['Container'],19.930027,19.853264,19.858965,19.858965,19.907882,19.907882,20.952565,39.664186,23.226362,21.272409,21.358491,22.030162,20.467375,37.835186,24.186759,22.181850,21.669850,22.780875,20.042218,41.877720,30.192411,26.397472,24.387328,23.818428,19.622952,44.327664,35.320875,33.096280,25.851035,25.644792,19.211404,45.068962,42.715708,42.876147,28.157199,26.992437,18.791978,44.483180,42.787410,45.669336,33.963459,27.502138,18.395903,39.244282,40.804824,43.297150,32.647299,27.839066,17.994761,35.175181,36.050385,42.620183,34.788411,28.057353,3693


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $1310.1m (23.1%), ... 

                              cifvalue    vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                              
10 | Cereals               428050351.0  1.165756e+09      7.6     15.6
03 | Fish and crustaceans  252454431.2  1.215589e+08      4.5      1.6
15 | Animal, vegetable or  125929937.2  1.409670e+08      2.2      1.9
17 | Sugars and sugar con  118803470.8  2.487604e+08      2.1      3.3
11 | Products of the mill   94443137.7  1.792792e+08      1.7      2.4
22 | Beverages, spirits a   78947957.7  6.832841e+07      1.4      0.9
04 | Dairy produce; birds   52692674.9  2.020885e+07      0.9      0.3
21 | Miscellaneous edible   45707324.7  1.103147e+07      0.8      0.1
23 | Food industries, res   39891145.4  6.347886e+07      0.7      0.9
19 | Preparations of cere   39381158.4  1.248165e+07      0.7      0.2
16 | Meat, fish, crustace    9801330.8  2.1315

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $865.6m (15.3%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Agriculture       497881100.0  732688284.7      8.8      9.8
Food & Beverages  345078062.6  409990086.9      6.1      5.5
Fishing            22637347.8   10848597.2      0.4      0.1



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $707.3m (12.5%), ... 

                                                 cifvalue   vol_kg_vtw  \
gtap_desc                                                                
PCR - Processed rice                          206297740.5  358339175.2   
OFD - Food products n.e.c.                    172889009.8  105689888.1   
VOL - Vegetable oils and fats                 146964555.4  190167255.1   
SGR - Sugar                                   106108569.3  233038827.5   
MIL - Dairy products                           53273403.3   20777899.9   
V_F - Vegetables, fruit, nuts                   6966367.7    9891897.7   
GRO - Cereal grains n.e.c.                      5585334.7   23006173.8   
OMT - Meat products n.e.c.                      4176849.9    1059476.5   
OCR - Crops n.e.c.                              3358893.6    1311878.8   
PFB - Plant-based fibers                        1004387.0    1921084.5   
CTL - Bovine cattle, sheep and goats, horses  

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 23.1% 
GTAP 12.5% 
EORA 15.3%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46350/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,120,Cameroon,826,United Kingdom,Import,Sea,392210,"392210 | Plastics; baths, sho",3922,3922 | Sanitary ware; baths,39,39 | Plastics and article,3693,"Baths, wash-basins, lavatory pans and covers, ...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",1249.929812,NaN,1249.929812,kg,3282.853,3282.853,208.238307,7896.0,4906.345416,1021.689061,['Container'],['Container'],3.257277,3.244731,3.245663,3.245663,3.253658,3.253658,3.424396,6.482543,3.796016,3.476670,3.490739,3.600514,3.345099,6.183619,3.952979,3.625305,3.541626,3.723207,3.275613,6.844313,4.934517,4.314288,3.985759,3.892781,3.207090,7.244721,5.772691,5.409113,4.224981,4.191274,3.139829,7.365876,6.981270,7.007492,4.601890,4.411527,3.071279,7.270138,6.992989,7.463998,5.550840,4.494830,3.006547,6.413915,6.668964,7.076298,5.335733,4.549896,2.940986,5.748879,5.891919,6.965658,5.685667,4.585572,3693,NaN
1,1,120,Cameroon,804,Ukraine,Import,Sea,392210,"392210 | Plastics; baths, sho",3922,3922 | Sanitary ware; baths,39,39 | Plastics and article,3693,"Baths, wash-basins, lavatory pans and covers, ...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",5883.985627,NaN,5883.985627,kg,7705.218,7705.218,980.272005,10263.0,6377.130573,6251.322576,['Container'],['Container'],19.930027,19.853264,19.858965,19.858965,19.907882,19.907882,20.952565,39.664186,23.226362,21.272409,21.358491,22.030162,20.467375,37.835186,24.186759,22.181850,21.669850,22.780875,20.042218,41.877720,30.192411,26.397472,24.387328,23.818428,19.622952,44.327664,35.320875,33.096280,25.851035,25.644792,19.211404,45.068962,42.715708,42.876147,28.157199,26.992437,18.791978,44.483180,42.787410,45.669336,33.963459,27.502138,18.395903,39.244282,40.804824,43.297150,32.647299,27.839066,17.994761,35.175181,36.050385,42.620183,34.788411,28.057353,3693,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $789.6m of $5665.9m (13.9%)
Volume: 1628.5 of 7461.4 million kilograms (21.8%)
Value-to-Weight: 0.48 from 0.76 US$ per kg (+-36.2%)
Average distance to market (volume-weighted): 6388.7 from 6642.3 nm (+-3.8%)

            food_group  N_com     cifvalue   vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    789  369153727.9  530029136.9      6.5      7.1
1  Unprocessed Cereals     54  221741186.9  807396836.4      3.9     10.8
2      Sugar and Honey    132  118804757.1  248760806.6      2.1      3.3
3                Dairy    239   52332469.3   20033414.5      0.9      0.3
4        Fats and Oils    131   13054746.5    8047286.1      0.2      0.1
5           Vegetables    336   11739945.0   11841480.4      0.2      0.2
6               Fruits    186    1760352.1    1693914.5      0.0      0.0
7        Meat and Prep     52     993605.8     736914.8      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,4.090378e+07,7.737675e+07,4.533297e+07,4.152635e+07,4.169422e+07,4.300466e+07,3.750928e+07,8.794699e+07,8.334661e+07,8.367284e+07,5.495690e+07,5.268154e+07,3.513700e+07,6.864240e+07,7.034789e+07,8.315749e+07,6.788042e+07,5.475829e+07
1,Fats and Oils,1.862014e+07,3.508230e+07,2.061330e+07,1.889628e+07,1.897218e+07,1.956955e+07,1.708641e+07,3.990213e+07,3.779679e+07,3.796896e+07,2.498587e+07,2.394473e+07,1.601426e+07,3.116512e+07,3.194979e+07,3.775246e+07,3.082103e+07,2.488313e+07
2,Fruits,1.950005e+06,3.642093e+06,2.153026e+06,1.978602e+06,1.986214e+06,2.047803e+06,1.792100e+06,4.150591e+06,3.928519e+06,3.951541e+06,2.608050e+06,2.499118e+06,1.681606e+06,3.255047e+06,3.333472e+06,3.929081e+06,3.205513e+06,2.596574e+06
3,Meat and Prep,1.500222e+06,2.839991e+06,1.663028e+06,1.523123e+06,1.529287e+06,1.577379e+06,1.375554e+06,3.226978e+06,3.058482e+06,3.069970e+06,2.016080e+06,1.932682e+06,1.288441e+06,2.518574e+06,2.581239e+06,3.051643e+06,2.490881e+06,2.008931e+06
4,Processed Cereals,1.500772e+09,2.834095e+09,1.662429e+09,1.523343e+09,1.529475e+09,1.577582e+09,1.376654e+09,3.222233e+09,3.053816e+09,3.066532e+09,2.015697e+09,1.931366e+09,1.289913e+09,2.516379e+09,2.578696e+09,3.047229e+09,2.488045e+09,2.007299e+09
5,Sugar and Honey,4.062488e+08,7.190432e+08,4.416759e+08,4.095500e+08,4.110468e+08,4.244200e+08,3.768027e+08,8.323963e+08,7.909359e+08,8.018167e+08,5.421377e+08,5.078812e+08,3.562321e+08,6.606836e+08,6.742873e+08,7.866992e+08,6.508489e+08,5.253543e+08
6,Unprocessed Cereals,1.483765e+09,2.802844e+09,1.643747e+09,1.506068e+09,1.512143e+09,1.559749e+09,1.360980e+09,3.186611e+09,3.020461e+09,3.032755e+09,1.993504e+09,1.909627e+09,1.275185e+09,2.488389e+09,2.549964e+09,3.013413e+09,2.460717e+09,1.984660e+09
7,Vegetables,2.298080e+07,4.263310e+07,2.532194e+07,2.329544e+07,2.338468e+07,2.411601e+07,2.114609e+07,4.867150e+07,4.606509e+07,4.640989e+07,3.068281e+07,2.936418e+07,1.986031e+07,3.817208e+07,3.909597e+07,4.604908e+07,3.763919e+07,3.049825e+07


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.70,0.08,0.01,0.02,0.04,0.96,0.88,0.88,0.33,0.29,0.64,0.67,0.92,0.63,0.37
1,Fats and Oils,1.26,0.15,0.02,0.03,0.07,1.75,1.59,1.60,0.61,0.53,1.16,1.22,1.67,1.13,0.68
2,Fruits,0.96,0.12,0.02,0.02,0.06,1.34,1.21,1.23,0.46,0.40,0.89,0.94,1.28,0.87,0.52
3,Meat and Prep,1.35,0.16,0.02,0.03,0.08,1.86,1.69,1.71,0.64,0.56,1.24,1.30,1.77,1.21,0.73
4,Processed Cereals,3.61,0.44,0.06,0.08,0.21,5.00,4.54,4.58,1.73,1.50,3.32,3.49,4.76,3.25,1.94
5,Sugar and Honey,2.63,0.30,0.03,0.04,0.15,3.83,3.49,3.58,1.39,1.10,2.56,2.68,3.62,2.48,1.42
6,Unprocessed Cereals,5.95,0.72,0.10,0.13,0.34,8.23,7.48,7.54,2.85,2.47,5.47,5.75,7.84,5.35,3.20
7,Vegetables,1.67,0.20,0.03,0.03,0.10,2.34,2.12,2.15,0.81,0.70,1.56,1.64,2.23,1.51,0.91
